# Agent Harnesses: Same Model, Different Wrapper, Different Result
**Companion notebook for the Medium post — "The Same Model, Three Agents"**

This notebook demonstrates the central claim of Harness-Bench (arXiv 2605.27922): the *harness* — not the model — can dominate agent performance. We fix a simple scripted "model" policy and change only the harness around it, then compare success rate and token spend on a set of toy tool-use tasks.

Run it top to bottom — no dependencies beyond the standard library.


## 1. The toy environment: a workspace, tools, and tasks
An agent task: given a workspace of files, produce an artifact (a summary report) that must satisfy an output contract.


In [ ]:
import random, json, math

random.seed(42)

# ---- Workspace: fake files with numbers ----
workspace = {}
for i in range(40):
    workspace[f"data/file_{i:02d}.txt"] = [round(random.uniform(0, 100), 2) for _ in range(40)]

def oracle(task_files):
    "Ground truth: sum of all numbers in the given files."
    return round(sum(v for f in task_files for v in workspace[f]), 2)

# ---- Tasks: pick 8 random subsets of files ----
all_files = list(workspace.keys())
tasks = [sorted(random.sample(all_files, k=random.randint(8, 14))) for _ in range(8)]
print("Example task files:", tasks[0])
print("Oracle answer:", oracle(tasks[0]))


## 2. The "model" — fixed for all harnesses
A deliberately simple scripted policy (stand-in for an LLM): it reads file contents one tool call at a time, then writes an answer. The *policy never changes* — only the harness does.


In [ ]:
class ScriptedModel:
    "Fixed policy: list files, read each target file, then emit the sum."
    def __init__(self):
        self.known = {}  # file -> contents the model has 'read'

    def decide(self, target_files):
        unseen = [f for f in target_files if f not in self.known]
        if unseen:
            return ("read", unseen[0])
        total = round(sum(v for f in target_files for v in self.known[f]), 2)
        return ("answer", total)


## 3. Two harnesses, one model
- **NaiveHarness**: feeds the *entire* tool history into every model call (context bloat), no iteration cap, no verification.
- **HardenedHarness**: truncates old tool results as context fills (the arXiv 2608.26218 trick), a "loop breaker" iteration cap with stall detection, and an output-contract verifier.


In [ ]:
class NaiveHarness:
    def __init__(self, model, max_steps=200):
        self.model, self.max_steps = model, max_steps
        self.tokens = 0
    def run(self, target_files):
        history = []                      # grows unbounded
        for _ in range(self.max_steps):
            action, payload = self.model.decide(target_files)
            if action == "read":
                result = workspace[payload]
                self.model.known[payload] = result
                history.append(f"read {payload} -> {result}")
                self.tokens += len(f"read {payload} -> {result}".split()) + sum(len(h.split()) for h in history)
            else:
                self.tokens += len(str(payload).split())
                return payload            # no verification!
        return None

class HardenedHarness:
    def __init__(self, model, max_steps=60, ctx_budget=400):
        self.model, self.max_steps = model, max_steps
        self.ctx_budget, self.tokens = ctx_budget, 0
    def _compact(self, history):
        "Shorten older tool results as context fills (mechanical, no re-tuning)."
        if sum(len(h.split()) for h in history) <= self.ctx_budget or len(history) < 3:
            return history
        # keep the two most recent results full; shorten everything older
        return [h.split("->")[0] + " -> [...]" for h in history[:-2]] + history[-2:]
    def run(self, target_files):
        history, last_actions = [], []
        for step in range(self.max_steps):
            history = self._compact(history)
            action, payload = self.model.decide(target_files)
            last_actions.append((action, payload))
            # loop breaker: re-reading an already-read file = stalling
            if action == "read" and payload in self.model.known:
                stalls = sum(1 for a, p in last_actions if a == "read" and p in self.model.known)
                if stalls >= 3:
                    break
            if action == "read":
                result = workspace[payload]
                self.model.known[payload] = result
                history.append(f"read {payload} -> {result}")
                self.tokens += len(f"read {payload} -> {result}".split()) + sum(len(h.split()) for h in history)
            else:
                # output-contract verifier: answer must be a number in plausible range
                ok = isinstance(payload, (int, float)) and 0 <= payload <= 100 * 40 * len(target_files)
                self.tokens += len(str(payload).split())
                return payload if ok else None
        return None


## 4. The experiment: same model, different harness
Run both harnesses on all 8 tasks and compare success rate vs. oracle and total tokens spent.


In [ ]:
def evaluate(harness_cls, **kw):
    successes, tokens, steps = 0, 0, []
    for t in tasks:
        h = harness_cls(ScriptedModel(), **kw)
        ans = h.run(t)
        successes += (ans == oracle(t))
        tokens += h.tokens
    return successes, tokens

s_naive, tok_naive = evaluate(NaiveHarness)
s_hard,  tok_hard  = evaluate(HardenedHarness)

print(f"{'harness':<16}{'success':<10}{'tokens':<10}")
print(f"{'naive':<16}{s_naive}/8     {tok_naive:<10}")
print(f"{'hardened':<16}{s_hard}/8     {tok_hard:<10}")
print(f"\nToken savings from context management: {(1 - tok_hard/max(tok_naive,1))*100:.1f}%")


## 5. Takeaway
The model policy is byte-for-byte identical in both runs. What changed:
1. **Context management** — truncating stale tool outputs kept the working context within budget and cut token spend.
2. **Stopping rules** — the loop breaker prevents infinite "read" loops, the classic agent failure mode.
3. **Output-contract verification** — the answer is checked against a contract before it counts.

This mirrors the 2026 findings: Harness-Bench's 23.8-point best-vs-worst harness gap, the 28%→49% F2PF lift from pure context management (arXiv 2608.26218), and Self-Harness's 33–60% gains from letting the harness patch itself. Build the kitchen, not just the chef.
